# Tự viết vòng sinh token với Hugging Face Transformers

`model.generate()` giấu ba thứ mà AI Engineer cần nhìn thấy: model chỉ trả **logits** cho token tiếp theo, vòng
lặp sinh từng token là code bình thường, và **KV cache** quyết định tốc độ. Bạn viết vòng `decode` tách khỏi
model (test bằng logits giả), hai hàm bọc model Hugging Face (không cache và có KV cache), hai thước đo
(`fertility` của tokenizer, `distinct_n` cho độ đa dạng), rồi chạy thật với `Qwen/Qwen2.5-0.5B-Instruct` trên CPU.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, chạy, in bảng; test bỏ qua chúng.
- Lần đầu tải khoảng 1 GB từ Hugging Face (model) và vài MB (tokenizer Phi-3). Cần bộ `iwslt15-en-vi`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/09-llms-and-open-models/exercises/intermediate/01-hf-generate -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from collections.abc import Callable, Sequence

import numpy as np

## 1. Vòng sinh token

`decode(next_logits, prompt_ids, max_new_tokens, pick, eos_ids=()) -> list[int]`:

- Mỗi bước gọi `next_logits(ids)` với **toàn bộ** chuỗi hiện tại (prompt + token đã sinh, kiểu `list[int]`); hàm
  trả logits (mảng 1 chiều, một số mỗi token của vocab) cho vị trí tiếp theo.
- Chọn token bằng `pick(logits, generated)`; `generated` là danh sách token đã sinh **trước** bước này.
- Thêm token vào kết quả. Token thuộc `eos_ids` thì vẫn thêm rồi dừng. Tối đa `max_new_tokens` token.
- Trả các token mới sinh (không gồm prompt), kiểu `int` Python. Không sửa `prompt_ids`.

`hf_next_logits(model)`: trả hàm `ids -> np.ndarray` gọi `model(input_ids=...)` trên **cả chuỗi** mỗi lần (tensor
`torch.long` shape `(1, n)`, trong `torch.no_grad()`), lấy logits vị trí cuối, đổi sang `float32` numpy.

`cached_next_logits(model)`: như trên nhưng dùng **KV cache**: giữ `past_key_values` và chuỗi đã xử lý giữa các
lần gọi. Nếu `ids` mới dài hơn và **bắt đầu bằng** chuỗi đã xử lý, chỉ đưa phần token mới vào model kèm
`past_key_values=` cache cũ và `use_cache=True`; ngược lại (chuỗi khác, hoặc lần đầu) chạy lại từ đầu với
`past_key_values=None`. Lưu `out.past_key_values` cho lần sau.

`import torch` đặt **trong** hàm, để import notebook không kéo torch khi chưa cần.

In [ ]:
def decode(next_logits: Callable[[list[int]], np.ndarray], prompt_ids: Sequence[int], max_new_tokens: int,
           pick: Callable[[np.ndarray, list[int]], int], eos_ids: Sequence[int] = ()) -> list[int]:
    """Sinh tối đa max_new_tokens token; dừng sau token eos (có trong kết quả)."""
    ids = list(prompt_ids)
    out: list[int] = []
    for _ in range(max_new_tokens):
        logits = next_logits(list(ids))
        tok = int(pick(logits, list(out)))
        out.append(tok)
        ids.append(tok)
        if tok in eos_ids:
            break
    return out


def hf_next_logits(model) -> Callable[[list[int]], np.ndarray]:
    """Không cache: mỗi lần chạy model trên cả chuỗi, trả logits vị trí cuối (float32 numpy)."""
    import torch

    def fn(ids: list[int]) -> np.ndarray:
        with torch.no_grad():
            out = model(input_ids=torch.tensor([list(ids)], dtype=torch.long))
        return out.logits[0, -1].float().numpy()

    return fn


def cached_next_logits(model) -> Callable[[list[int]], np.ndarray]:
    """Có KV cache: chỉ đưa token mới vào model khi chuỗi nối tiếp chuỗi đã xử lý."""
    import torch

    seen: list[int] = []
    past = None

    def fn(ids: list[int]) -> np.ndarray:
        nonlocal seen, past
        ids = list(ids)
        if past is not None and len(ids) > len(seen) and ids[: len(seen)] == seen:
            new = ids[len(seen):]
        else:
            new, past = ids, None
        with torch.no_grad():
            out = model(input_ids=torch.tensor([new], dtype=torch.long), past_key_values=past, use_cache=True)
        seen, past = ids, out.past_key_values
        return out.logits[0, -1].float().numpy()

    return fn

## 2. Hai thước đo

`fertility(count_tokens, texts) -> float`: tổng số token (theo `count_tokens(text)`) chia tổng số **từ** (tách
bằng `str.split()`) trên mọi văn bản. Tiếng Việt: một "từ" ở đây là một âm tiết.

`distinct_n(texts, n) -> float`: số n-gram **khác nhau** chia tổng số n-gram, n-gram tạo từ `str.split()` của
**từng** văn bản (không nối qua ranh giới hai văn bản), gộp trên mọi văn bản. Không có n-gram nào thì trả `0.0`.
Đo độ đa dạng của nhiều câu trả lời sinh cho cùng một prompt: 1.0 là không lặp n-gram nào.

In [ ]:
def fertility(count_tokens: Callable[[str], int], texts: Sequence[str]) -> float:
    """Token trên mỗi từ (tách khoảng trắng), tính gộp."""
    tokens = sum(count_tokens(t) for t in texts)
    words = sum(len(t.split()) for t in texts)
    return tokens / words


def distinct_n(texts: Sequence[str], n: int) -> float:
    """Tỉ lệ n-gram khác nhau trên tổng n-gram, n-gram không vượt ranh giới văn bản."""
    grams = [tuple(w[i:i + n]) for w in (t.split() for t in texts) for i in range(len(w) - n + 1)]
    return len(set(grams)) / len(grams) if grams else 0.0

In [ ]:
# explore: tìm gốc repo, nạp tokenizer và model Qwen2.5-0.5B-Instruct trên CPU (lần đầu tải khoảng 1 GB)
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

torch.set_num_threads(4)
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"nạp {MODEL} trong {time.perf_counter() - t0:.1f} s, {n_params:,} tham số, "
      f"{n_params * 4 / 1024**3:.2f} GiB ở float32 (checkpoint lưu bf16: {n_params * 2 / 1024**3:.2f} GiB)")
print("vocab:", len(tok), "| eos:", tok.eos_token, model.generation_config.eos_token_id,
      "| sampling mặc định trên model card:", {k: getattr(model.generation_config, k)
                                                for k in ["do_sample", "temperature", "top_p", "top_k", "repetition_penalty"]})

In [ ]:
# explore: chat template biến message thành chuỗi nào, và phân phối token đầu tiên của câu trả lời
messages = [{"role": "user", "content": "Thủ đô của Việt Nam là thành phố nào? Trả lời một câu."}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_ids = tok(prompt)["input_ids"]
print(repr(prompt))
print(len(prompt_ids), "token prompt")

logits = hf_next_logits(model)(prompt_ids)
probs = torch.softmax(torch.from_numpy(logits).double(), dim=-1).numpy()
order = np.argsort(-probs)
entropy = float(-(probs[probs > 0] * np.log(probs[probs > 0])).sum())
print(f"entropy token đầu: {entropy:.3f} nat")
for i in order[:8]:
    print(f"{tok.decode([int(i)])!r:>14}  p = {probs[i]:.4f}")
mass = np.cumsum(probs[order])
print("khối xác suất của top-1 / top-5 / top-20:", [round(float(mass[k - 1]), 4) for k in (1, 5, 20)])

In [ ]:
# explore: greedy 4 cách: vòng của bạn (không cache, có KV cache) và model.generate (mặc định, tắt penalty)
N_NEW = 48
greedy_pick = lambda logits, generated: int(np.argmax(logits))
eos = model.generation_config.eos_token_id
eos = eos if isinstance(eos, list) else [eos]

timings, outs = {}, {}
for name, fn in [("decode không cache", hf_next_logits(model)), ("decode có KV cache", cached_next_logits(model))]:
    t0 = time.perf_counter()
    outs[name] = decode(fn, prompt_ids, N_NEW, greedy_pick, eos)
    timings[name] = time.perf_counter() - t0
# generate() đọc generation_config.json của model: do_sample=False tắt sampling nhưng repetition_penalty 1.1 vẫn áp
for name, extra in [("generate (mặc định)", {}), ("generate, penalty 1.0", {"repetition_penalty": 1.0})]:
    t0 = time.perf_counter()
    with torch.no_grad():
        gen = model.generate(torch.tensor([prompt_ids]), max_new_tokens=N_NEW, do_sample=False, top_p=None,
                             top_k=None, temperature=None, **extra)
    timings[name] = time.perf_counter() - t0
    outs[name] = gen[0, len(prompt_ids):].tolist()

ref = outs["generate, penalty 1.0"]
for name in outs:
    n = len(outs[name])
    print(f"{name:22s} {n:3d} token, {timings[name]:6.2f} s, {n / timings[name]:5.1f} token/s, "
          f"giống generate penalty 1.0: {outs[name] == ref} | {tok.decode(outs[name], skip_special_tokens=True)!r}")

In [ ]:
# explore: chi phí không cache tăng theo độ dài: thời gian mỗi bước khi chuỗi dài 64, 256, 1024 token
fn_plain = hf_next_logits(model)
for n in (64, 256, 1024):
    ids = (prompt_ids * (n // len(prompt_ids) + 1))[:n]
    fn_plain(ids)                                        # khởi động
    plain = []
    for _ in range(5):
        t0 = time.perf_counter()
        fn_plain(ids)
        plain.append(time.perf_counter() - t0)
    fn_cache = cached_next_logits(model)
    fn_cache(ids)                                        # nạp cache cho n token
    cached = []
    for k in range(5):
        t0 = time.perf_counter()
        fn_cache(ids + [11] * (k + 1))                   # mỗi lần chỉ thêm 1 token mới
        cached.append(time.perf_counter() - t0)
    print(f"chuỗi {n:5d} token: không cache {np.median(plain) * 1000:7.1f} ms/bước, "
          f"có cache {np.median(cached) * 1000:6.1f} ms/bước (trung vị 5 lần)")

In [ ]:
# explore: temperature và độ đa dạng: 8 câu trả lời mỗi mức, top_p 0.9, seed ghim
creative = [{"role": "user", "content": "Viết một câu mở đầu cho truyện ngắn về Hà Nội mùa thu."}]
c_ids = tok.apply_chat_template(creative, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"]
for temp in (0.3, 0.7, 1.0, 1.5):
    torch.manual_seed(0)
    with torch.no_grad():
        g = model.generate(c_ids, do_sample=True, temperature=temp, top_p=0.9, top_k=0, max_new_tokens=40,
                           num_return_sequences=8)
    texts = tok.batch_decode(g[:, c_ids.shape[1]:], skip_special_tokens=True)
    print(f"T={temp}: distinct-1 {distinct_n(texts, 1):.3f}, distinct-2 {distinct_n(texts, 2):.3f} | {texts[0][:70]!r}")

In [ ]:
# explore: fertility trên 1 268 cặp câu IWSLT15 tst2013 (cùng nội dung, tiếng Anh và tiếng Việt)
import html

import tiktoken

IWSLT = ensure("iwslt15-en-vi")


def clean(line: str) -> str:
    """Bỏ thực thể HTML và khoảng trắng trước dấu câu mà bộ dữ liệu đã tách sẵn."""
    text = html.unescape(line.strip())
    for p in [" ,", " .", " ?", " !", " :", " ;", " 's", " n't", " '"]:
        text = text.replace(p, p[1:])
    return text


en = [clean(x) for x in (IWSLT / "tst2013.en").read_text(encoding="utf-8").splitlines()]
vi = [clean(x) for x in (IWSLT / "tst2013.vi").read_text(encoding="utf-8").splitlines()]
assert len(en) == len(vi) == 1268
phi3 = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
counters = {
    "Qwen2.5 (151k vocab)": lambda t: len(tok(t, add_special_tokens=False)["input_ids"]),
    "Phi-3 / Llama 2 (32k)": lambda t: len(phi3(t, add_special_tokens=False)["input_ids"]),
    "tiktoken cl100k_base": lambda t, e=tiktoken.get_encoding("cl100k_base"): len(e.encode(t)),
    "tiktoken o200k_base": lambda t, e=tiktoken.get_encoding("o200k_base"): len(e.encode(t)),
}
for name, count in counters.items():
    f_en, f_vi = fertility(count, en), fertility(count, vi)
    t_en, t_vi = sum(map(count, en)), sum(map(count, vi))
    print(f"{name:24s} token/từ: en {f_en:.3f}, vi {f_vi:.3f} | tổng token en {t_en:6d}, vi {t_vi:6d}, "
          f"vi/en {t_vi / t_en:.2f}, ký tự/token vi {sum(map(len, vi)) / t_vi:.2f}")

## Đọc kết quả

- Bốn cách greedy có cho cùng chuỗi token không? `generate` mặc định khác ở đâu, vì sao (gợi ý: đọc
  `generation_config.json` ở cell đầu)?
- KV cache nhanh hơn bao nhiêu ở chuỗi 64 và 1 024 token? Vì sao khoảng cách lớn dần theo độ dài?
- Temperature tăng thì distinct-2 thay đổi thế nào? Ở T = 1.5 câu còn đọc được không?
- Cùng một nội dung, tiếng Việt tốn gấp bao nhiêu token so với tiếng Anh với từng tokenizer? Điều đó nghĩa là gì
  cho chi phí API tính theo token và cho context window?

Sau đó làm phần trắc nghiệm của bài trong app.